In this notebook, I'll try 10 random numbers and get the optimal random number for saebm and also ucl gmm. Because DEBM does not output the likelihood, I cannot decide which random number to use. 

**It turns out UCL GMM results do not change with seeds.**

In [2]:
import pandas as pd 
import numpy as np 
from pysaebm import run_ebm
from typing import List, Dict, Tuple, Optional
import copy 
from collections import Counter
from scipy import stats
# Go up one level
# from the current working directory
import os 
import sys 
import altair as alt 
# Get current working directory (where notebook is run)
current_dir = os.getcwd()

# Move two levels up
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path if not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import run  # Now this should work!
import utils_adni
import json 
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns 
import yaml 

from pyebm import debm
from pyebm import ebm
from typing import List, Dict, Tuple, Optional
from kde_ebm import mixture_model
from kde_ebm import mcmc
from kde_ebm.mixture_model import get_prob_mat
from scipy.stats import kendalltau
from collections import defaultdict, namedtuple
import yaml
from kde_ebm.plotting import mcmc_uncert_mat

In [3]:
meta_data = ['PTID', 'DX_bl', 'VISCODE', 'COLPROT']

select_biomarkers = ['MMSE_bl', 'Ventricles_bl', 'WholeBrain_bl', 
            'MidTemp_bl', 'Fusiform_bl', 'Entorhinal_bl', 
            'Hippocampus_bl', 'ADAS13_bl', 'PTAU_bl', 
            'TAU_bl', 'ABETA_bl', 'RAVLT_immediate_bl', 'ICV_bl'
]

diagnosis_list = ['CN', 'EMCI', 'LMCI', 'AD']

OUTPUT_DIR = parent_dir

raw = f'{OUTPUT_DIR}/ADNIMERGE.csv'

In [4]:
def load_config():
    config_path = os.path.join(parent_dir, "config.yaml")
    
    with open(config_path, "r") as f:
        return yaml.safe_load(f)
config = load_config()
print("Loaded config:")
print(json.dumps(config, indent=4))

Loaded config:
{
    "N_VARIANTS": 50,
    "NStartpoints": 25,
    "Niterations": 1000,
    "N_MCMC_UCL": 100000,
    "N_MCMC": 10000,
    "N_SHUFFLE": 2,
    "BURN_IN": 500,
    "THINNING": 1,
    "GEN_SEED": 42,
    "ADNI_MCMC_SEED": 4790,
    "JS": [
        50,
        200,
        500,
        1000
    ],
    "MCMC_SEED": 53,
    "RS": [
        0.1,
        0.25,
        0.5,
        0.75,
        0.9
    ],
    "N_BOOTSTRAP": 50,
    "SA_EBM_ALGO_NAMES": [
        "conjugate_priors",
        "mle",
        "kde",
        "em",
        "hard_kmeans"
    ],
    "OTHER_ALGO_NAMES": [
        "debm",
        "debm_gmm",
        "ucl_gmm",
        "ucl_kde"
    ],
    "OUTPUT_DIR": "algo_results",
    "EXPERIMENT_NAMES": [
        "sn_kjOrdinalDM_xnjNormal",
        "sn_kjOrdinalDM_xnjNonNormal",
        "sn_kjOrdinalUniform_xnjNormal",
        "sn_kjOrdinalUniform_xnjNonNormal",
        "sn_kjContinuousUniform",
        "sn_kjContinuousBeta",
        "xiNearNormal_kjContinuousUnifor

In [5]:
# Number of independent optimization attempts in greedy ascent
NStartpoints=config['NStartpoints']
Niterations=config['Niterations']
N_MCMC=config['N_MCMC']
# N_MCMC = 100
N_SHUFFLE=config['N_SHUFFLE']
BURN_IN=config['BURN_IN']
# BURN_IN = 10
THINNING=config['THINNING']
N_BOOTSTRAP=config['N_BOOTSTRAP']
MCMC_SEED = config['MCMC_SEED']
ADNI_MCMC_SEED = config['ADNI_MCMC_SEED']

# Algo names
sa_ebm_algo_names = config['SA_EBM_ALGO_NAMES']
other_algo_names = config['OTHER_ALGO_NAMES']

In [6]:
adni_filtered = utils_adni.get_adni_filtered(raw, meta_data, select_biomarkers, diagnosis_list)
debm_output, data_matrix, df_long, participant_dx_dict, ordered_biomarkers = utils_adni.process_data(
    adni_filtered, ventricles_log=False, tau_log=False)

726
No duplicates!
AD: 153 (21.1%)
LMCI: 236 (32.5%)
CN: 155 (21.3%)
EMCI: 182 (25.1%)
----------------------------------------------------
ADNI1: 275 (37.9%)
ADNI2: 375 (51.7%)
ADNIGO: 76 (10.5%)


In [7]:
# # Just for one run, check what's happening during MCMC
# np.random.seed(42)
# X, y, feature_names = data_matrix[:, :-1], data_matrix[:, -1].astype(int), np.array(ordered_biomarkers)
# mixture_models = mixture_model.fit_all_gmm_models(X, y)

# # Check the res output more carefully
# res = mcmc.mcmc(X, mixture_models, n_iter=100_000,
#                 greedy_n_init=10,
#                 greedy_n_iter=1000)

# print(f"Total MCMC samples: {len(res)}")
# print(f"First 5 orderings:")
# for i in range(5):
#     print(f"  Sample {i}: {res[i].ordering}, likelihood: {res[i].score}")

# print(f"Last 5 orderings:")
# for i in range(-5, 0):
#     print(f"  Sample {len(res)+i}: {res[i].ordering}, likelihood: {res[i].score}")

# # Check unique orderings and likelihoods
# unique_orderings = len(set(tuple(sample.ordering) for sample in res))
# unique_likelihoods = len(set(sample.score for sample in res))

# print(f"\nUnique orderings: {unique_orderings}")
# print(f"Unique likelihoods: {unique_likelihoods}")

# # Likelihood statistics
# all_likes = [sample.score for sample in res]
# print(f"Min likelihood: {min(all_likes)}")
# print(f"Max likelihood: {max(all_likes)}")
# print(f"Std of likelihoods: {np.std(all_likes)}")

## For SAEBM

In [8]:
dic = defaultdict(float)
for _ in range(10):
    x = np.random.randint(1, 2**32 - 1)
    results = run_ebm(
            data_file=os.path.join(parent_dir, 'adni.csv'),
            algorithm='conjugate_priors',
            output_dir=OUTPUT_DIR,
            n_iter=20000,
            n_shuffle=2,
            burn_in=500,
            thinning=1,
            skip_heatmap=False,
            skip_traceplot=False,
            seed=x, 
            save_results=True,
            save_details=True,
            save_theta_phi=True,
        )
    dic[x] = results['max_log_likelihood']

2025-10-09 14:54:24,234 - INFO - Running conjugate priors for file: adni
2025-10-09 14:54:24,238 - INFO - Number of biomarkers: 12
2025-10-09 14:54:47,309 - INFO - Iteration 2000/20000, Acceptance Ratio: 3.75%, Log Likelihood: 286.0084, 
2025-10-09 14:54:54,565 - INFO - Iteration 4000/20000, Acceptance Ratio: 3.42%, Log Likelihood: 289.3633, 
2025-10-09 14:55:01,843 - INFO - Iteration 6000/20000, Acceptance Ratio: 3.23%, Log Likelihood: 285.1142, 
2025-10-09 14:55:09,024 - INFO - Iteration 8000/20000, Acceptance Ratio: 3.81%, Log Likelihood: 285.0874, 
2025-10-09 14:55:16,224 - INFO - Iteration 10000/20000, Acceptance Ratio: 3.60%, Log Likelihood: 287.3571, 
2025-10-09 14:55:23,479 - INFO - Iteration 12000/20000, Acceptance Ratio: 3.00%, Log Likelihood: 287.3571, 
2025-10-09 14:55:30,694 - INFO - Iteration 14000/20000, Acceptance Ratio: 2.57%, Log Likelihood: 287.3571, 
2025-10-09 14:55:37,890 - INFO - Iteration 16000/20000, Acceptance Ratio: 2.25%, Log Likelihood: 287.3571, 
2025-10-0

```py
defaultdict(float,
            {
             42: 284,
             2223108049: 290.06628056461716,
             1839970752: 309.3491341249183,
             2910885818: 290.0890856314178,
             1263692870: 170.58932040153107,
             2267309967: 258.1687269492893,
             2489319570: 290.86353774283316,
             1132025739: 282.2956452011818,
             3107182077: 308.611931716077,
             3750502537: 309.59130012697335,
             3684580103: 258.437455353994})
```

## For UCL

In [10]:
# # Generate all seeds upfront with a fixed seed for reproducibility
# np.random.seed(12345)  # Master seed for generating experiment seeds
# seeds = np.random.randint(1, 2**32 - 1, size=10)

# dic = defaultdict(float)
# for i, seed in enumerate(seeds):
#     print(f"Run {i+1}/10 with seed {seed}")
    
#     # Set seed for this specific run
#     np.random.seed(seed)
    
#     X, y, feature_names = data_matrix[:, :-1], data_matrix[:, -1].astype(int), np.array(ordered_biomarkers)
#     mixture_models = mixture_model.fit_all_gmm_models(X, y)
#     res = mcmc.mcmc(X, mixture_models, n_iter=100_000,
#                     greedy_n_init=25,
#                     greedy_n_iter=1000)
#     ml_stages = run.ebm_staging(
#         x=X,
#         mixtures=mixture_models,
#         samples=res
#     )
#     res.sort(reverse=True)
#     best_likelihood = res[0].score
#     dic[seed] = best_likelihood

In [11]:
# dic 